# Remap YOLO Class Labels

This notebook remaps class numbers in YOLO format label files.

## Overview
The notebook:
1. Loads label files from the specified directory
2. Maps classes from the pre-swap list to the post-swap list
3. Updates all bounding box annotations with new class numbers
4. Saves the modified labels back to the original files

## Use Cases
- Consolidate multiple class labels into a single class
- Reindex class numbers for model compatibility
- Merge or reorganize class hierarchies

## Import Required Libraries

In [2]:
import os
from CellProcessor import (
    use_dataset,
    list_dataset,
)

In [3]:
"""List available datasets and select one to use."""
print("Available datasets:")
list_dataset()

Available datasets:


,ID,Cell_type,Death_type,Image_path,Description
0,1,MEF,Necroptosis,Data,Test


## Configure Class Mapping

Define the mapping rules for remapping classes:
- **classes_pre_swap**: Original class numbers (grouped in lists)
- **classes_post_swap**: Target class numbers to map to

## Notes
- This script modifies label files in-place; ensure you have backups if needed
- The YOLO label format: `<class_id> <x_center> <y_center> <width> <height>` (normalized coordinates)
- Only the class ID (first column) is modified; bounding box coordinates remain unchanged
- To map multiple classes to one, add them to the same sublist: `[[0, 1, 2]]`
- To map to different classes, use multiple sublists: `[[0], [1]]` → `[2, 3]`

In [10]:

# Load dataset (using preset 1; modify as needed)
dataset = use_dataset(1)
print(f"Dataset: {dataset['Cell_type']} cells, {dataset['Death_type']} death type")
print(f"Image path: {dataset['Image_path']}\n")

# Construct input paths (source data)
base_output = os.path.join(dataset['Image_path'], dataset['Death_type'])
LABELED_IMAGES_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Labeled_phase_aug/")

# Define class mapping: classes_pre_swap[i] -> classes_post_swap[i]
# Example: [0,1] -> 1 means classes 0 and 1 both become class 1
# Example: [0] -> 1 means classes 0  become class 1

classes_pre_swap = [[0]]  # List of lists of classes to remap
classes_post_swap = [1]   # List of target class numbers

print(f"Labels directory: {LABELED_IMAGES_DIR_PATH}")
print(f"Mapping rules:")
for i, pre_classes in enumerate(classes_pre_swap):
    print(f"  {pre_classes} → {classes_post_swap[i]}")

Dataset: MEF cells, Necroptosis death type
Image path: Data

Labels directory: Data/Necroptosis/MEF_Labeled_phase_aug/
Mapping rules:
  [0] → 1


## Process Label Files

Iterate through all label files and apply the class remapping.

In [11]:
# Get all label files
labels = os.listdir(LABELED_IMAGES_DIR_PATH)
i_label = 1
len_labels = len(labels)

print(f"Total label files: {len_labels}")
print("Processing started...\n")

# Process each label file in directory
for label in labels:
    # Print progress every 100 files
    if i_label % 100 == 0:
        print(f"Progress: {i_label}/{len_labels}")
    i_label += 1

    new_lines = []
    label_path = os.path.join(LABELED_IMAGES_DIR_PATH, label)
    
    # Read the label file
    try:
        with open(label_path, 'r') as label_file:
            lines = label_file.readlines()
    except Exception as e:
        print(f"Error reading {label}: {e}")
        continue
    
    # Process each bounding box line
    for line in lines:
        new_line = line
        line_class = line.split(" ")[0]
        
        # Check each class mapping rule
        for classes_list_id in range(len(classes_pre_swap)):
            classes_pre = classes_pre_swap[classes_list_id]
            
            # Check if current line's class matches any class in the pre-swap list
            for class_pre in classes_pre:
                if int(line_class) == class_pre:
                    # Replace class number with new class number
                    new_line = str(classes_post_swap[classes_list_id]) + line[len(line_class):]
        
        new_lines.append(new_line)
    
    # Write the modified lines back to the file
    try:
        with open(label_path, 'w') as label_file_write:
            label_file_write.writelines(new_lines)
    except Exception as e:
        print(f"Error writing {label}: {e}")

print(f"\n✓ Processing complete! Updated {len_labels} label files.")

Total label files: 3042
Processing started...

Progress: 100/3042
Progress: 200/3042
Progress: 300/3042
Progress: 400/3042
Progress: 500/3042
Progress: 600/3042
Progress: 700/3042
Progress: 800/3042
Progress: 900/3042
Progress: 1000/3042
Progress: 1100/3042
Progress: 1200/3042
Progress: 1300/3042
Progress: 1400/3042
Progress: 1500/3042
Progress: 1600/3042
Progress: 1700/3042
Progress: 1800/3042
Progress: 1900/3042
Progress: 2000/3042
Progress: 2100/3042
Progress: 2200/3042
Progress: 2300/3042
Progress: 2400/3042
Progress: 2500/3042
Progress: 2600/3042
Progress: 2700/3042
Progress: 2800/3042
Progress: 2900/3042
Progress: 3000/3042

✓ Processing complete! Updated 3042 label files.
